# Sparse autoencoders: recovery against baselines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/sparse_autoencoder.ipynb)

Self-contained CPU experiment. Install missing dependencies with `%pip install torch numpy scipy matplotlib`. Retain Colab's supplied PyTorch build. Tested on Python 3.12, PyTorch 2.14.0+rocm10.1.0, NumPy 2.5.3, SciPy 1.18.1 and Matplotlib 3.11.2. The ROCm suffix identifies the development-machine GPU build.

In [ ]:
import math, time
import numpy as np
import torch
from torch.nn import functional as F
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
torch.set_num_threads(4)
from torch import nn
from scipy.optimize import linear_sum_assignment

## Sparseae

Eight observed coordinates expand through a ReLU encoder to 24 non-negative coefficients. The linear decoder has unit-norm columns, preventing coefficient shrinkage through arbitrary decoder scaling.

In [ ]:
class SparseAE(nn.Module):

    def __init__(self, d=8, width=24):
        super().__init__()
        self.encoder = nn.Linear(d, width)
        self.decoder = nn.Linear(width, d, bias=False)
        with torch.no_grad():
            self.decoder.weight.copy_(F.normalize(self.decoder.weight, dim=0))

    def forward(self, x):
        z = F.relu(self.encoder(x))
        return (self.decoder(z), z)

## Sparse data

Twelve planted directions are mixed into eight observed dimensions. Each feature is present independently with probability 0.08, so expected true sparsity is 0.96 active features and many observations are all zero.

In [ ]:
def sparse_data(seed, n=4096):
    g = torch.Generator().manual_seed(seed)
    dictionary = F.normalize(torch.randn(12, 8, generator=g), dim=1)
    features = (torch.rand(n, 12, generator=g) < 0.08) * (0.5 + torch.rand(n, 12, generator=g))
    return (features @ dictionary, features, dictionary)

## Match dictionary

Use only live decoder units. Signed cosine respects non-negative coefficients. Hungarian matching gives each true feature a distinct eligible unit; absolute matching is retained only as a diagnostic.

In [ ]:
def match_dictionary(dictionary, decoder, activations):
    # Exclude inactive units before matching; coefficients cannot change sign.
    live = torch.where(activations.max(0).values > 0.0001)[0]
    cosine = dictionary @ F.normalize(decoder, dim=0)
    scores = cosine[:, live].numpy()
    rows, columns = linear_sum_assignment(-scores)
    signed = np.zeros(len(dictionary))
    signed[rows] = scores[rows, columns]
    abs_rows, abs_columns = linear_sum_assignment(-np.abs(scores))
    absolute = np.zeros(len(dictionary))
    absolute[abs_rows] = np.abs(scores[abs_rows, abs_columns])
    units = np.full(len(dictionary), -1)
    units[rows] = live.numpy()[columns]
    return dict(signed=float(signed.mean()), absolute=float(absolute.mean()), recovered=int((signed > 0.9).sum()), live=int(len(live)), units=units.tolist(), signed_scores=signed.tolist(), cosine=cosine.tolist())

## Sae experiment

Fit four penalties with three shared initialisation seeds. Compare signed live-unit matching with untrained decoders and random dictionaries having the same eligible width. A bounded stopping diagnostic is recorded for each arm. Finally remove a genuinely nonzero live coefficient and check the reconstruction change.

In [ ]:
def sae_experiment():
    start = time.perf_counter()
    x, features, dictionary = sparse_data(118)
    train, test = (x[:3072], x[3072:])
    records, traces, initial = ([], [], [])
    example = None
    # Cache random baselines by eligible width, so comparisons use equal choices.
    baseline_cache = {}
    rng = np.random.default_rng(118)
    for seed in [0, 1, 2]:
        for penalty in [0.0, 0.05, 0.15, 0.5]:
            torch.manual_seed(seed)
            model = SparseAE()
            with torch.no_grad():
                _, initial_codes = model(test)
                initial_match = match_dictionary(dictionary, model.decoder.weight, initial_codes)
            if penalty == 0:
                initial.append(dict(seed=seed, **initial_match))
            opt = torch.optim.Adam(model.parameters(), lr=0.003)
            history, checkpoint_losses = ([], [])
            stable = False
            for step in range(5000):
                idx = torch.randint(len(train), (256,))
                reconstructed, codes = model(train[idx])
                mse = (reconstructed - train[idx]).square().mean()
                loss = mse + penalty * codes.mean()
                opt.zero_grad()
                loss.backward()
                opt.step()
                with torch.no_grad():
                    model.decoder.weight.copy_(F.normalize(model.decoder.weight, dim=0))
                    if (step + 1) % 100 == 0:
                        full_recon, full_codes = model(train)
                        objective = float((full_recon - train).square().mean() + penalty * full_codes.mean())
                        history.append([step + 1, float((full_recon - train).square().mean()), float((full_codes > 0.0001).float().sum(1).mean()), objective])
                        checkpoint_losses.append(objective)
                        if step + 1 >= 2000 and len(checkpoint_losses) >= 11:
                            recent = abs(checkpoint_losses[-1] - checkpoint_losses[-6]) / max(abs(checkpoint_losses[-6]), 1e-09)
                            prior = abs(checkpoint_losses[-6] - checkpoint_losses[-11]) / max(abs(checkpoint_losses[-11]), 1e-09)
                            if recent < 0.01 and prior < 0.01:
                                stable = True
                                break
            with torch.no_grad():
                recon, codes = model(test)
                matched = match_dictionary(dictionary, model.decoder.weight, codes)
                live_count = matched['live']
                if live_count not in baseline_cache:
                    values = []
                    for _ in range(500):
                        random = rng.normal(size=(8, live_count))
                        random /= np.linalg.norm(random, axis=0)
                        cos = dictionary.numpy() @ random
                        rr, cc = linear_sum_assignment(-cos)
                        values.append(float(cos[rr, cc].sum() / 12))
                    baseline_cache[live_count] = dict(mean=float(np.mean(values)), low=float(np.quantile(values, 0.025)), high=float(np.quantile(values, 0.975)), draws=500)
                recent_change = abs(checkpoint_losses[-1] - checkpoint_losses[-6]) / max(abs(checkpoint_losses[-6]), 1e-09)
                rec = dict(seed=seed, penalty=penalty, mse=float((recon - test).square().mean()), mean_active=float((codes > 0.0001).float().sum(1).mean()), dead=24 - live_count, dead_units=torch.where(codes.max(0).values <= 0.0001)[0].tolist(), recovery=matched['signed'], absolute_recovery=matched['absolute'], recovered_above_point9=matched['recovered'], random_baseline=baseline_cache[live_count], updates=step + 1, criterion_met=stable, last_500_relative_change=recent_change, initial_signed_recovery=initial_match['signed'])
                records.append(rec)
                traces.append(dict(seed=seed, penalty=penalty, history=history))
                # Choose a live unit and actually recompute the edited reconstruction.
            if seed == 0 and penalty == 0.5:
                    feature = int(np.argmax(matched['signed_scores']))
                    unit = matched['units'][feature]
                    sample = int(codes[:, unit].argmax())
                    original = recon[sample]
                    edited_codes = codes[sample].clone()
                    coefficient = float(edited_codes[unit])
                    assert coefficient > 0.0001
                    edited_codes[unit] = 0
                    ablated = model.decoder(edited_codes)
                    difference = ablated - original
                    expected = -coefficient * model.decoder.weight[:, unit]
                    assert torch.allclose(difference, expected, atol=1e-06)
                    example = dict(cos=matched['cosine'], matched_units=matched['units'], feature=feature, unit=unit, coefficient=coefficient, signed_cosine=matched['signed_scores'][feature], sample=sample, original=original.tolist(), ablated=ablated.tolist(), difference=difference.tolist(), expected_difference=expected.tolist(), input=test[sample].tolist(), true_feature_amplitude=float(features[3072 + sample, feature]), ablation_l2=float(difference.norm()))
    return dict(records=records, traces=traces, example=example, initial_baselines=initial, true_mean_active=float((features[3072:] > 0).float().sum(1).mean()), zero_input_fraction=float((features[3072:] == 0).all(1).float().mean()), expected_true_active=0.96, stop_rule='Two consecutive 500-update objective changes below 1% after 2000 updates; maximum5000', seconds=time.perf_counter() - start)

## Run and print the complete comparison

All measurements below are calculated in this notebook. Fixed seeds fix samples, while hardware and library changes can affect timing and floating-point details.

In [ ]:
result = sae_experiment()
def print_table(result, topic):
    """Print every experimental row, followed by the article's grouped means."""
    print('penalty mse active signed_match abs_match features_above_0.9 random_mean updates criterion_met')
    for penalty in [0.0, 0.05, 0.15, 0.5]:
        rows = [row for row in result['records'] if row['penalty'] == penalty]
        keys = ['mse', 'mean_active', 'recovery', 'absolute_recovery', 'recovered_above_point9']
        print(penalty, *[float(np.mean([row[key] for row in rows])) for key in keys], np.mean([row['random_baseline']['mean'] for row in rows]), [row['updates'] for row in rows], [row['criterion_met'] for row in rows])
    print('Untrained live-unit signed matches:', [row['signed'] for row in result['initial_baselines']])
    print('True held-out active features:', result['true_mean_active'], 'Zero inputs:', result['zero_input_fraction'])
    print('Live-unit ablation:', {key: value for key, value in result['example'].items() if key != 'cos'})
print_table(result, 'sae')

## Sae plots

The figure uses the newly computed result, with baselines and units labelled.

In [ ]:
def sae_plots(result):
    figure, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for penalty in [0., .05, .15, .5]:
        rows = [row for row in result['records'] if row['penalty'] == penalty]
        axes[0].scatter([row['mean_active'] for row in rows], [row['mse'] for row in rows], label=str(penalty), s=45)
        axes[1].scatter([penalty]*3, [row['recovery'] for row in rows], s=45)
    axes[0].axvline(result['true_mean_active'], color='grey', linestyle='--', label='True active features')
    axes[0].set(xlabel='Mean active units', ylabel='Held-out reconstruction MSE', yscale='log')
    axes[0].legend(title='Penalty', fontsize=9)
    baseline = np.mean([row['random_baseline']['mean'] for row in result['records']])
    initial = np.mean([row['signed'] for row in result['initial_baselines']])
    axes[1].axhline(baseline, color='grey', linestyle='--', label='Matched-width random mean')
    axes[1].axhline(initial, color='black', linestyle=':', label='Untrained live-unit mean')
    axes[1].set(xlabel='Sparsity penalty', ylabel='Signed live-unit matched cosine', ylim=(.45, 1))
    axes[1].legend(fontsize=9)
    figure.tight_layout()
    return figure
sae_plots(result)
plt.show()

## Sae dictionary

The figure uses the newly computed result, with baselines and units labelled.

In [ ]:
def sae_dictionary(result):
    example = result['example']
    matched = example['matched_units']
    matrix = np.array(example['cos'])[:, matched]
    figure, axis = plt.subplots(figsize=(7, 5))
    image = axis.imshow(matrix, vmin=-1, vmax=1, cmap='coolwarm')
    axis.set(xlabel='Live decoder units ordered by true-feature match', ylabel='True feature', title='Seed 0, penalty 0.5; signed cosine')
    figure.colorbar(image, ax=axis, shrink=.8)
    figure.tight_layout()
    return figure
sae_dictionary(result)
plt.show()

## Sae intervention

The figure uses the newly computed result, with baselines and units labelled.

In [ ]:
def sae_intervention(result):
    example = result['example']
    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    coordinate = np.arange(8)
    axes[0].bar(coordinate-.18, example['original'], width=.36, label='Reconstruction')
    axes[0].bar(coordinate+.18, example['ablated'], width=.36, label='Live unit removed')
    axes[1].scatter(coordinate, example['difference'], label='Measured change', s=50)
    axes[1].scatter(coordinate, example['expected_difference'], marker='x', label='−coefficient × decoder', s=65)
    for axis in axes:
        axis.set(xlabel='Activation coordinate', ylabel='Value')
        axis.legend(fontsize=10)
    figure.suptitle(f"Live unit {example['unit']}, coefficient {example['coefficient']:.3f}; no sign flip")
    figure.tight_layout()
    return figure
sae_intervention(result)
plt.show()

## Animated experiment

The seed-zero penalty-0.5 trajectory shows both reconstruction MSE and active units. The final frame is held for two seconds.

In [ ]:
def progress_animation(result, topic):
    figure, axes = plt.subplots(1, 2, figsize=(10, 4))
    frames = list(range(15)) + [14] * 6

    def update(frame):
        fraction = (frame + 1) / 15
        for axis in axes:
            axis.clear()
        history = next((row['history'] for row in result['traces'] if row['seed'] == 0 and row['penalty'] == 0.5))
        count = max(1, int(fraction * len(history)))
        rows = history[:count]
        axes[0].plot([row[0] for row in rows], [row[1] for row in rows], 'o-')
        axes[1].plot([row[0] for row in rows], [row[2] for row in rows], 'o-')
        axes[0].set(ylabel='Training MSE', yscale='log', ylim=(min((row[1] for row in history)) * 0.8, max((row[1] for row in history)) * 1.2))
        axes[1].set(ylabel='Mean active units', ylim=(0, max((row[2] for row in history)) * 1.2))
        axes[1].axhline(result['expected_true_active'], color='grey', linestyle='--', label='True expected active = 0.96')
        axes[1].legend(fontsize=8)
        for axis in axes:
            axis.set(xlabel='Training update', xlim=(50, history[-1][0] + 100))
        figure.suptitle('Seed 0, penalty 0.5; reconstruction and sparsity')
        figure.tight_layout()
        return []
    return FuncAnimation(figure, update, frames=frames, interval=333, blit=False)
from IPython.display import Image, display
animation = progress_animation(result, 'sae')
animation.save('sae_progress.gif', writer=PillowWriter(fps=3))
plt.close(animation._fig)
display(Image(filename='sae_progress.gif'))

## Exercises and limits

Compare feature frequencies and hold the eligible-width random baseline fixed appropriately. Inspect all per-seed recovered-feature counts. Run the live-unit edit in a downstream model before making behavioural claims.

## References

Toy Models of Superposition: https://transformer-circuits.pub/2022/toy_model/index.html; Towards Monosemanticity: https://transformer-circuits.pub/2023/monosemantic-features/index.html